**REWARD DiD pipeline — engine v20.58.** Paths: one `INPUT_DIR` in `_paths.py` (output, temp, panel, results, estimator files and the shipped `REWARD_ground_inputs/` derive from it; RAM, GPU memory and every core are used up to 98 % of the total — `MEMORY_SHARE` there is an optional manual split, 1.0 = none). The design — timing (your fund workbook by default), dose, rings, years, seasons, fragments, overlaps, fixed effects, covariates — is set in each notebook's CELL 1 and applied when it runs (what CELL 1 leaves unset comes from P00's saved defaults). Multi-site: `08_Multisite_Runs/MS01`. Which models the data supports: P10. Pre-built packages: `R/rstudio/R_RUN_ALL_MODELS.Rmd`, `python_prebuilt/`. Details: the bundle's `docs/CHANGELOG_v20.md`, `PIPELINES_GUIDE.md`, `PREBUILT_MODEL_MAP.md`.

# MS01 -- Multi-site runs: one model, every sub-watershed, then all 20 pooled
The pooled export carries **SWSiD_All** (1..20); `data/sites/sites.csv` (from the shapefile `SWSs20_KarnatakaAll5k`) gives each site its name, phase (Saturation Ph 1: 11 sites, Ph 2: 9 sites) and treatment year. This notebook runs ONE chosen model per site (each site on its own treatment year) and once pooled (site x period fixed effects, clusters by site, two cohorts) -- still one model at a time, as you run them today.

Fill `treatment_year` per phase in `_sites.PHASE_TREATMENT_YEAR` or per site in `sites.csv` before the pooled run.

In [ ]:
# ---- ROBUST IMPORT BOOTSTRAP (v9.0 fix for "ModuleNotFoundError: No module named '_common'") ----
# Jupyter's working directory is NOT always the notebook's folder, so a bare
# sys.path.insert(0,".") can fail. This searches the CWD, its parents, and their
# sub-folders for _common.py and reports exactly what it found.
import sys, os, glob as _glob

def _locate(fname, max_up=4):
    seen, cands = set(), []
    d = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if d in seen: break
        seen.add(d)
        cands.append(d)
        cands += [p for p in _glob.glob(os.path.join(d, "*")) if os.path.isdir(p)]
        parent = os.path.dirname(d)
        if parent == d: break
        d = parent
    for c in cands:
        if os.path.exists(os.path.join(c, fname)):
            return c
    return None

_dir = _locate("_common.py")
if _dir is None:
    print("[FAILED]  Could not find _common.py")
    print(f"[INFO]    Jupyter's working directory is: {os.getcwd()}")
    print("[INFO]    Fix by ONE of these:")
    print("[INFO]      a) open this notebook from inside the folder that contains _common.py, OR")
    print("[INFO]      b) set the full path manually in the next line and re-run:")
    print("[INFO]         sys.path.insert(0, r'C:\\path\\to\\All_DIDsCodes')")
    raise ModuleNotFoundError("_common.py not found -- see the guidance above")
if _dir not in sys.path:
    sys.path.insert(0, _dir)
print(f"[OK]      found _common.py in: {_dir}")

import _common as C      # v20.29: the engine is imported here (it was never imported before first use)

# ---- CELL 1: configuration ----
C.require_engine("20.58")
C.start_cell_log("MS01")   # v20.36: every message of this notebook also goes to <OUTPUT_DIR>\cell_logs
import _sites as S
MODEL_ID  = "M01"                      # the model to run per site + pooled (its estimation is defined in CELL 2)
OUTCOME   = C.current_outcome("NDVI")
# ---- v20.57: the design of these runs -- applied HERE at run time, as in every model's CELL 1 (P00 is never re-run for it)
DESIGN_MODE    = "recommended"         # "recommended": an option set to "data" is chosen from the data | "manual"
TREATMENT_TIMING = "fund"              # "fund": each site's first treated season from your fund workbook | "registry" | "fixed"
TREATMENT_YEAR = 2022                  # "fixed" timing, and the fall-back for a site the fund file / registry does not date
FUND_START_RULE = "backcast"           # "backcast" | "share" | "file_start"
DOSE_VARIABLE  = "dose_intensity_per_ha"
CONTROL_ZONES  = "data"                # "data" | "1-5" | "1-3" ...
CLUSTER        = "site"                # v20.59: "site" (the sub-watersheds; fewer than 6 -> the years) | "block" (~1 km spatial blocks of pixels: many clusters, spatial correlation absorbed)
PRE_YEARS      = "data"; POST_YEARS = "data"     # "data" | "all" | a number of years | 2015 (a calendar year = the FIRST pre year)
SEASONS        = "all"
EXCLUDE_TRANSITION_YEAR = False        # True = the first treated year of each series leaves the sample (robustness)
OVERLAP_ROWS   = "drop"; FRAGMENT_RULE = "drop"
UNIT_FE        = "pixel_season"        # one fixed effect per pixel x season series | "pixel"
POOLED_FE      = "site_period"         # the POOLED run: sub-watershed x year x season effects | "period" (each single-site run: the same)
EXCLUDE_GAPFILLED = True               # rows the exporter filled from history are not observations | False keeps them
OUTCOME_SCREEN    = "drop"             # v20.59: "drop" (a year-season constant across pixels -- a fill value -- or with collapsed coverage leaves the
                                       #   model; its evidence: OUTCOME_SCREEN_<outcome>.csv beside the results) | "keep" (reported and KEPT; results tagged
                                       #   _screenKept) | "off". A v20.58 run stopped here ("41 year-seasons are NOT pixel data ... Re-export it")
DESIGN_SOURCE     = "panel"            # v20.59 -- YOUR RULE: the PANEL's treat / control / pre / post / did (the exports' Treat flag, PERIOD_RULE) are
                                       #   what this model estimates on (results tagged _panelDesign; the design in effect above is COMPARED with them) | "model" =
                                       #   design-based modelling: the timing / TREATMENT_YEAR / transition-year settings above build post / pre / did
COVARIATES     = "all"                 # "all" (the four weather covariates) | "mean_temp_rain" | "weather" | "none" | [list]
SITES     = None                       # None = every site in the panel (after the fragment rule); or e.g. [1, 7, 8]
USE_SITE_YEARS = True                  # with TREATMENT_TIMING = "fixed": True = each site on its registry year (the v20.22 switch)
# ================================================================================================================================
# OPTIONAL CUSTOMISATIONS (1 Oct, your rule) -- every one has a USE_ switch. False = NOT applied at all, whatever the value beside it
# says (the standard design above runs as it is); True = applied exactly as set. All are False by default. DESIGN IN EFFECT shows each
# one as "not used" or with the value in force; a rule in force also tags the results folder (the tag is named beside each option).
# --------------------------------------------------------------------------------------------------------------------------------
# SECTION B -- THE CONTROL GROUP CHOSEN ON THE PRE PERIOD (your request of 30 Sep): which buffers / clusters are the controls for THIS
# outcome, decided on the pre period only and then FIXED across the whole panel (the same control pixels in every year and season)
# --------------------------------------------------------------------------------------------------------------------------------
USE_CONTROL_SELECTION = False    # False = every ring of CONTROL_ZONES is the control group (as above) | True = CONTROL_SELECTION below chooses it
CONTROL_SELECTION = "pre_rings"  #   "pre_rings": the CONTROL_SELECT_K (1 or 2) buffers whose PRE-period series is closest to the treatment area's -- your
                                 #   "most close 1 or 2 buffers" | "pre_blocks": control CLUSTERS (~1 km blocks of pixels) from ANY part of the buffers, the
                                 #   closest first, until CONTROL_SELECT_RATIO x the treated pixels -- your "clusters from any part of the buffers".
                                 #   Per outcome; the decision uses the PRE period only (a post- or outcome-based rule is refused); made once and applied
                                 #   to every year and season. Evidence: CONTROL_SELECTION_<outcome>.csv beside the results. Tag _ctrlPre2r / _ctrlPreBlk3x
CONTROL_SELECT_K  = 2            #   pre_rings: how many buffers -- 1 or 2 (up to 5)
CONTROL_SELECT_RATIO = 3.0       #   pre_blocks: control pixels >= this x the treated pixels
CONTROL_SELECT_ON = "trend"      #   what "closest" means: "trend" = the demeaned pre series' distance (what the parallel-trends assumption asks; recommended)
                                 #   | "level" = the nearest pre-period MEAN (your "mean of the treatment area vs the buffer's mean") | "rmse" = the RMSE of the
                                 #   cell-wise pre differences | "both" = trend + level.  Tag suffix: none / L / R / B
USE_SAME_PIXELS   = False        # False = a pixel may contribute to one side only (the v20.58 sample) | True = the treated and control groups are the SAME
SAME_PIXELS       = "pre_post"   #   pixels across the panel: "pre_post" = every pixel observed in pre AND post, else it leaves (tag _pixPP) | "all" = in
                                 #   every year-season of the sample (a balanced pixel set; tag _pixAll). The sample-integrity line confirms it at every run
USE_BALANCED_PANEL = False      # 4 Oct (your request): False = every pixel is kept where values are missing (an unbalanced panel, as before) |
BALANCED_PANEL    = "drop"       #   True = applied: "drop" = per variable, a pixel missing the value in ANY year-season of the sample leaves this
                                 #   model's sample (a balanced panel; tag _balanced) | "keep" = kept and counted. The panel file keeps every row
# --------------------------------------------------------------------------------------------------------------------------------
# SECTION C -- PANEL-PREPARATION ENHANCEMENTS (the three specifications of 30 Sep); each with its own switch
# --------------------------------------------------------------------------------------------------------------------------------
USE_DONUT         = False        # False = ring 1 is a control like the others | True = DONUT_RINGS leave the control pool -- the spillover buffer next to the core
DONUT_RINGS       = [1]          #   the ring(s) left out, e.g. [1] or [1, 2]; applied before any control choice. Tag _donut1
USE_LANDUSE_MASK  = False        # False = every land-use class | True = only pixels whose PRE-period (baseline) LandUse class is in LANDUSE_KEEP stay. Tag _lu2
LANDUSE_KEEP      = [2]          #   your exporter's class code(s) for agriculture (a pixel's class is read on the pre period, so the works cannot move it)
USE_BASELINE_NDVI_MASK = False   # False = no mask | True = only pixels whose PRE-period mean NDVI exceeds BASELINE_NDVI_MIN stay (an agricultural mask). Tag _ndviPre0.25
BASELINE_NDVI_MIN = 0.25         #   the threshold (an NDVI value)
USE_COVERAGE_THRESHOLD = False   # False = the standard screen (a year-season below 5 % of the typical coverage leaves) | True = below MIN_PIXEL_COVERAGE_PCT. Tag _cov70
MIN_PIXEL_COVERAGE_PCT = 0.70    #   the share of the typical treated / control coverage a year-season must reach (your log: 2023 Zaid, 2025 Kharif / Zaid collapsed)
USE_DROP_SINGLETONS = False      # False = series seen once stay (pyfixest drops them itself) | True = they leave before the demeaning (the pre-flight). Tag _noSingle
USE_PRECISION_TOLERANCE = False  # False = the no-data zero is an EXACT 0 (the v20.58 rule) | True = |value| <= PRECISION_TOLERANCE is the no-data zero and a
PRECISION_TOLERANCE = 1e-6       #   year-season is "constant across pixels" within it -- indices in [-1, 1] are never compared with exact equality
# The Rabi-only run of the specifications is SEASONS = "Rabi" above; the ~1 km block clusters are CLUSTER = "block" above.
# ================================================================================================================================
C.set_scenario(design_mode=DESIGN_MODE, timing=TREATMENT_TIMING, treatment_year=TREATMENT_YEAR, fund_start_rule=FUND_START_RULE,
               dose_variable=DOSE_VARIABLE, control_zones=CONTROL_ZONES, pre_years=PRE_YEARS, post_years=POST_YEARS, seasons=SEASONS,
               exclude_transition_year=EXCLUDE_TRANSITION_YEAR, overlap_rows=OVERLAP_ROWS, fragment_rule=FRAGMENT_RULE,
               unit_fe=UNIT_FE, pooled_fe=POOLED_FE, exclude_gapfilled=EXCLUDE_GAPFILLED, covariates=COVARIATES,
               outcome_screen=OUTCOME_SCREEN, design_source=DESIGN_SOURCE,
               control_selection=CONTROL_SELECTION, control_select_k=CONTROL_SELECT_K, control_select_ratio=CONTROL_SELECT_RATIO, control_select_on=CONTROL_SELECT_ON, same_pixels=SAME_PIXELS, cluster=CLUSTER,
               donut_rings=DONUT_RINGS, landuse_keep=LANDUSE_KEEP, baseline_ndvi_min=BASELINE_NDVI_MIN, min_pixel_coverage_pct=MIN_PIXEL_COVERAGE_PCT, drop_singletons=USE_DROP_SINGLETONS, precision_tolerance=PRECISION_TOLERANCE,
               use_control_selection=USE_CONTROL_SELECTION, use_same_pixels=USE_SAME_PIXELS, use_balanced_panel=USE_BALANCED_PANEL, balanced_panel=BALANCED_PANEL, use_donut=USE_DONUT, use_landuse_mask=USE_LANDUSE_MASK,
               use_baseline_ndvi_mask=USE_BASELINE_NDVI_MASK, use_coverage_threshold=USE_COVERAGE_THRESHOLD, use_precision_tolerance=USE_PRECISION_TOLERANCE)
C.resolve_design()                     # DESIGN IN EFFECT (the pooled design; each site is resolved again on its own in CELL 3)
print(S.registry().to_string(index=False))
C.info(f"sites in the panel (after the fragment rule): {C.sites_in_panel(after_fragments=True)}")
# v20.28: one row per SWS -- name, phase, start year, treated / control pixels, and how its rows matched the shapefile.
# The 20 SWS are 20 CLUSTERS and strata; COHORTS are distinct start years (fill per-site years in data/sites/sites.csv
# if they differ: the staggered estimators M05/M09/M27/M31 switch on automatically with >= 2 cohorts).
SITE_TABLE = C.site_design_table(save_as=os.path.join(C.RESULTS_ROOT, "site_design_table.csv"))


In [ ]:
# ---- CELL 2: the model, exactly as the single-site notebook estimates it (M01 shown; swap in any model's run_model body) ----
def estimate(where):
    RESULTS_DIR = os.path.join(C.results_dir(MODEL_ID), where); os.makedirs(RESULTS_DIR, exist_ok=True)
    df = C.load_panel(columns=C.columns_for(OUTCOME))
    df = C.build_treatment_columns(df); df = df[df['in_analysis_sample'] == 1]
    C.treatment_coverage(df)
    beta, se = C.estimate_twfe_did(df, OUTCOME, 'did_term', 'pixel_id', 'time_fe_yearseason', 'subwshed_id', covariates=C.DEFAULT_COVARIATES)
    row = {'outcome': OUTCOME, 'where': where, 'beta': beta, 'se': se, 'treatment_year': C.ACTIVE['treatment_year'],
           'cluster': C.ACTIVE['cluster'], 'fe': C.ACTIVE['pooled_fe'], 'n_sites': df['site_id'].nunique() if 'site_id' in df else 1}
    C.save_results(row, RESULTS_DIR, f'canonical_twfe_{OUTCOME}.csv')
    return row


In [ ]:
# ---- CELL 3: run per site, then pooled ----
log = C.run_sites(estimate, sites=SITES, pooled=True, per_site=True, use_site_years=USE_SITE_YEARS, label=MODEL_ID, pooled_fe=POOLED_FE)
print(log.drop(columns=['result'], errors='ignore').to_string(index=False))


In [ ]:
# ---- CELL 4 (optional): the package-input tables for the pre-built models (python_prebuilt/pf_pipeline.py; R: the bundle's R/ pipeline) ----
C.SITE_FILTER = None
C.set_scenario(cluster='site', pooled_fe=POOLED_FE, **({'use_site_years': True} if USE_SITE_YEARS and TREATMENT_TIMING == 'fixed' else {}))     # pooled layout (the timing of CELL 1)
files, meta = C.export_for_packages(OUTCOME)
C.set_scenario(cluster='subwshed', pooled_fe='period')
